# Telegram Channel Crawl
<!-- structured-notebook -->
## Notebook Summary
Purpose: discover and scrape longevity-relevant Telegram channels using forward-based recursive crawling.

Main steps:
- Start from a set of seed channels known to discuss longevity and biohacking.
- Scrape each channel via the Telethon API, apply English-language and keyword-relevance filters.
- Extract `forwarded_from` metadata and discover new channels.
- Recurse to `MAX_DEPTH` depth; build a NetworkX DiGraph of forwarding relationships.
- Export graph as GEXF (Gephi-compatible) plus `nodes.json` / `edges.json`.
- Route all API traffic through a SOCKS5 proxy to handle rate limits.

### Outcome and limitation

After narrowing to stricter longevity keywords, only **LongevityInTime** and **DailyBiohacking** consistently pass relevance checks. English-speaking longevity channels are scarce on Telegram — the community is dominated by Russian, Ukrainian, Turkish, and Spanish-language channels. External directories (TGDataset, TGStat, prior research) yielded no useful English channel lists. This is an inherent limitation of the Telegram sub-project.


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `.env` (SEED_CHANNELS_1, API_ID, API_HASH, PHONE, DECODO_*, MAX_DEPTH) | — |
| Output | `<DATA_ROOT>/Telegram/output/messages.json` | Consumed by `02_exploration/01_data_exploration.ipynb`, `03_topic_modeling/01_topic_modeling.ipynb` |
| Output | `<DATA_ROOT>/Telegram/output/tg_channel_network.gexf` | Gephi-compatible graph |
| Output | `<DATA_ROOT>/Telegram/output/nodes.json`, `edges.json` | Graph components |


## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Walk up from the notebook's directory to the repo root (the first ancestor that contains `src/`) and add it to sys.path
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from src.shared_reddit_telegram.config import PROJECT_ROOT, TELEGRAM_PIPELINE, TELEGRAM_DATA

In [ ]:
import os
import json
import asyncio
from datetime import datetime
from collections import defaultdict

import networkx as nx
from dotenv import load_dotenv
from telethon import TelegramClient
from telethon.tl.functions.channels import JoinChannelRequest
from telethon.errors import (
    ChannelPrivateError,
    FloodWaitError,
    UsernameInvalidError,
    UsernameNotOccupiedError,
)

from src.shared_reddit_telegram.text_cleaning import is_english

## 2. Configuration

Credentials are loaded from the `.env` file at the project root. The crawl parameters:

- **SEED_CHANNELS**: Initial channels known to discuss longevity, biohacking, and health optimization
- **MAX_DEPTH**: How many forwarding hops to follow (depth 0 = seed channels only, depth 1 = channels forwarded to by seeds, etc.)
- **Proxy**: DECODO SOCKS5 proxy for Telegram API access

In [ ]:
load_dotenv(PROJECT_ROOT / ".env")

API_ID = int(os.getenv("API_ID"))
API_HASH = os.getenv("API_HASH")
PHONE = os.getenv("PHONE")
MAX_DEPTH = int(os.getenv("MAX_DEPTH", 2))

SEED_CHANNELS = json.loads(
    os.getenv("SEED_CHANNELS", '["LiveHealthy", "DailyBiohacking"]')
)

# DECODO SOCKS5 proxy configuration
PROXY = None
decodo_user = os.getenv("DECODO_USER")
decodo_pass = os.getenv("DECODO_PASSWORD")
if decodo_user and decodo_pass:
    import socks
    PROXY = (socks.SOCKS5, "gate.decodo.com", 7000, True, decodo_user, decodo_pass)

print(f"Seed channels: {SEED_CHANNELS}")
print(f"Max crawl depth: {MAX_DEPTH}")
print(f"Proxy configured: {PROXY is not None}")

## 3. Telethon Client Setup

In [ ]:
session_path = str(TELEGRAM_PIPELINE / "crawl_session")

client = TelegramClient(session_path, API_ID, API_HASH, proxy=PROXY)

await client.start(phone=PHONE)
print("Telethon client connected.")

## 4. Crawl Logic

The recursive depth-based approach works as follows:

1. At each depth level, iterate over the current frontier of channel usernames
2. For each channel, join it (if not already a member) and iterate through its messages
3. When a message has a `forwarded_from` attribute pointing to another channel, record that channel as a new discovery and add a directed edge to the graph
4. The newly discovered channels become the frontier for the next depth level
5. Already-visited channels are skipped to avoid cycles

In [ ]:
G = nx.DiGraph()
visited = set()
all_messages = []


async def scrape_channel(channel_username, message_limit=500):
    """Scrape messages from a single channel and return forwarded-from channels."""
    discovered = set()
    messages = []

    try:
        entity = await client.get_entity(channel_username)
        try:
            await client(JoinChannelRequest(entity))
        except Exception:
            pass  # May already be a member

        async for msg in client.iter_messages(entity, limit=message_limit):
            if msg.text:
                messages.append(
                    {
                        "source": channel_username,
                        "msg_id": msg.id,
                        "date": msg.date.isoformat() if msg.date else None,
                        "text": msg.text,
                        "forwarded_from": None,
                    }
                )

                # Extract forwarding information
                if msg.forward and msg.forward.chat:
                    fwd = msg.forward.chat
                    if hasattr(fwd, "username") and fwd.username:
                        fwd_username = fwd.username
                        messages[-1]["forwarded_from"] = fwd_username
                        discovered.add(fwd_username)
                        G.add_edge(channel_username, fwd_username)

    except (ChannelPrivateError, UsernameInvalidError, UsernameNotOccupiedError) as e:
        print(f"  Skipping {channel_username}: {type(e).__name__}")
    except FloodWaitError as e:
        print(f"  FloodWait: sleeping {e.seconds}s")
        await asyncio.sleep(e.seconds)

    return messages, discovered


async def crawl_depth(frontier, depth):
    """Crawl all channels at the given depth level."""
    print(f"\n--- Depth {depth} ({len(frontier)} channels) ---")
    next_frontier = set()

    for i, ch in enumerate(frontier):
        if ch in visited:
            continue
        visited.add(ch)
        G.add_node(ch, depth=depth)

        print(f"  [{i + 1}/{len(frontier)}] Scraping: {ch}")
        messages, discovered = await scrape_channel(ch)
        all_messages.extend(messages)

        new = discovered - visited
        next_frontier |= new
        print(f"    -> {len(messages)} messages, {len(new)} new channels discovered")

    return next_frontier

## 5. Run Crawl

Execute the crawl from depth 0 (seed channels) through `MAX_DEPTH`.

In [ ]:
frontier = set(SEED_CHANNELS)

for depth in range(MAX_DEPTH + 1):
    frontier = await crawl_depth(frontier, depth)
    if not frontier:
        print(f"\nNo new channels discovered at depth {depth}. Stopping.")
        break

print(f"\nCrawl complete.")
print(f"Total channels visited: {len(visited)}")
print(f"Total messages collected: {len(all_messages)}")
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## 6. Export Graph & Messages

Save the forwarding network as GEXF (Gephi-compatible), JSON node/edge lists, and the collected messages.

In [ ]:
output_dir = TELEGRAM_PIPELINE / "hybrid"
output_dir.mkdir(parents=True, exist_ok=True)

# Save network graph
nx.write_gexf(G, str(output_dir / "tg_channel_network.gexf"))

# Save nodes and edges as JSON
nodes = [{"id": n, **G.nodes[n]} for n in G.nodes]
edges = [{"source": u, "target": v} for u, v in G.edges]

with open(output_dir / "nodes.json", "w") as f:
    json.dump(nodes, f, indent=2)

with open(output_dir / "edges.json", "w") as f:
    json.dump(edges, f, indent=2)

# Save collected messages
with open(output_dir / "messages.json", "w") as f:
    json.dump(all_messages, f, indent=2, default=str)

print(f"Saved {len(nodes)} nodes, {len(edges)} edges to {output_dir}")
print(f"Saved {len(all_messages)} messages to messages.json")

## 7. Quick Network Summary

In [ ]:
# Top channels by in-degree (most forwarded FROM)
in_deg = sorted(G.in_degree(), key=lambda x: x[1], reverse=True)[:15]
print("Top 15 channels by in-degree (most forwarded from):")
for ch, deg in in_deg:
    print(f"  {ch}: {deg}")

In [ ]:
await client.disconnect()
print("Client disconnected.")

---

**Important:** This notebook requires a live Telegram API connection with the DECODO SOCKS5 proxy. Ensure the following environment variables are set in your `.env` file:

- `API_ID` — Telegram API application ID
- `API_HASH` — Telegram API application hash
- `PHONE` — Phone number associated with the Telegram account
- `DECODO_USER` — DECODO proxy username
- `DECODO_PASSWORD` — DECODO proxy password
- `SEED_CHANNELS` (optional) — JSON array of seed channel usernames
- `MAX_DEPTH` (optional, default: 2) — Maximum crawl depth

---
<!-- nav-footer -->

[Project Overview](../../../../PROJECT_OVERVIEW.ipynb) | [01 data exploration](../../../../SocialMedia/Telegram/notebooks/02_exploration/01_data_exploration.ipynb) →
